# Debate & Critique Pattern

*Notebook 15*

Challenge and revise agent output with separate proposer, critic, and judge roles.


---

## 🔧 Setup

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(dotenv_path=Path("..") / ".env")

from agents import Agent, Runner

# Notebook-specific imports
import asyncio

MODEL = "gpt-5-mini"
REASONING_MODEL = "gpt-5"

print("✅ Ready!")

---

## 🎯 The Problem

A single run returns one completed answer.

Without a review step, the workflow does not explicitly check for errors or weak reasoning.

Separate proposer and critic roles make critique and revision explicit.

---

## 💡 Part 1: The Pattern

The Debate & Critique pattern uses sequential `Runner.run()` calls, passing output forward at each step:

```
Topic
  ↓
Proposer   →  initial response
  ↓
Critic      →  feedback and weaknesses
  ↓
Proposer   →  revised response (using the critique)
  ↓
Final output
```

Each agent receives the previous agent's `result.final_output` as part of its input.

No special SDK feature is required.

---

## ✍️ Part 2: Basic Proposer / Critic

A writer drafts.

An editor critiques.

The writer revises.

In [ ]:
writer_instructions = (
    "You write clear, concise explanations of technical concepts.\n"
    "Target audience: beginner developers. Use plain language and one concrete example."
)

writer_agent = Agent(
    name="Writer",
    instructions=writer_instructions,
    model=MODEL
)

editor_instructions = (
    "You are a critical technical editor. Review the given explanation and identify:\n"
    "1. Any unclear or jargon-heavy language\n"
    "2. Missing or weak examples\n"
    "3. Logical gaps or oversimplifications\n"
    "4. What specifically should be improved\n"
    "Be direct and specific. Do not rewrite, only critique."
)

editor_agent = Agent(
    name="Editor",
    instructions=editor_instructions,
    model=MODEL
)

# --------------------------------------------------------------
print("✅ Writer and Editor agents ready")

#### Step 1: First Draft

In [ ]:
topic = "Explain what an API is"

print("✍️ STEP 1: FIRST DRAFT")
print("=" * 60)

draft_result = await Runner.run(writer_agent, input=topic)

first_draft = draft_result.final_output

print(first_draft)
print("=" * 60)

#### Step 2: Critique

In [ ]:
print("🔍 STEP 2: CRITIQUE")
print("=" * 60)

critique_input = f"Please critique this explanation:\n\n{first_draft}"

critique_result = await Runner.run(editor_agent, input=critique_input)

critique = critique_result.final_output

print(critique)
print("=" * 60)

#### Step 3: Revised Draft

In [ ]:
print("✅ STEP 3: REVISED DRAFT")
print("=" * 60)

revision_input = (
    f"Here is your original explanation:\n\n{first_draft}\n\n"
    f"Here is the editor's critique:\n\n{critique}\n\n"
    f"Please rewrite the explanation addressing all the feedback."
)

revision_result = await Runner.run(writer_agent, input=revision_input)

revised_draft = revision_result.final_output

print(revised_draft)
print("=" * 60)

### 💡 Key Takeaway

The critic has one job: find weaknesses.

The writer then has specific, targeted feedback to act on rather than vague instructions to "improve."

---

## 🔄 Part 3: The Critique Loop, With Termination Rules

A quality gate repeats scoring, critique, and revision.

Stop when the judge score reaches the threshold or `max_iterations`.

Always cap the loop.

With `max_iterations=3`, the cap path can make 11 calls: 4 writer, 3 critic, and 4 judge.

This demo assigns `REASONING_MODEL` to the judge.

The writer and editor use the cheaper `MODEL`.

The judge has no ground truth here, so its score is a heuristic.

In [ ]:
from pydantic import BaseModel, Field
from typing import Annotated

class QualityScore(BaseModel):
    score: Annotated[int, Field(ge=1, le=10)]

judge_instructions = (
    "You evaluate explanation quality on a scale of 1-10.\n"
    "Criteria:\n"
    "- Clarity (is it easy to understand?)\n"
    "- Accuracy (is it technically correct?)\n"
    "- Example quality (is the example concrete and helpful?)\n\n"
    "Evaluate the explanation and return your score."
)

# The judge uses REASONING_MODEL because rubric scoring is judgment-heavy.
quality_judge_agent = Agent(
    name="QualityJudge",
    instructions=judge_instructions,
    model=REASONING_MODEL,
    output_type=QualityScore
)

# --------------------------------------------------------------
print("✅ QualityJudge agent ready")

#### Define the Critique Loop

In [ ]:
async def critique_loop(topic, quality_threshold=8, max_iterations=3):
    """Refine a draft until the judge score reaches the threshold or the cap.

    Returns (draft, final_score, termination_reason). The judge score is a
    heuristic, not ground truth, so a threshold-met draft is not "approved".
    """

    print(f"🔄 CRITIQUE LOOP: {topic}")
    print(f"    Target score: {quality_threshold}/10 | Max iterations: {max_iterations}")
    print("=" * 60)

    # Initial draft
    draft_result = await Runner.run(writer_agent, input=topic)
    current_draft = draft_result.final_output
    print(f"\n📝 Initial draft ({len(current_draft)} chars)")

    final_score = None
    termination_reason = None
    for iteration in range(1, max_iterations + 1):
        # Score the current draft
        score_result = await Runner.run(quality_judge_agent, input=f"Rate this explanation:\n\n{current_draft}")
        score = score_result.final_output.score
        print(f"\n🔄 Iteration {iteration}: Judge score = {score}/10")

        if score >= quality_threshold:
            print(f"✅ Judge score reached the threshold, stopping at iteration {iteration}")
            final_score = score
            termination_reason = "threshold_met"
            break

        # Critique and revise
        critique_result = await Runner.run(editor_agent, input=f"Critique this explanation:\n\n{current_draft}")
        critique = critique_result.final_output
        revision_result = await Runner.run(writer_agent, input=f"Original:\n{current_draft}\n\nCritique:\n{critique}\n\nRewrite addressing all feedback.")
        current_draft = revision_result.final_output
        print(f"    Revised to {len(current_draft)} chars")

    if termination_reason is None:
        # Score the final revision so every returned draft has a judge score
        score_result = await Runner.run(quality_judge_agent, input=f"Rate this explanation:\n\n{current_draft}")
        final_score = score_result.final_output.score
        if final_score >= quality_threshold:
            termination_reason = "threshold_met"
            print(f"\n✅ Final judge score reached the threshold at the iteration cap: {final_score}/10")
        else:
            termination_reason = "max_iterations"
            print(f"\n⚠️ Max iterations reached. Final revision scored {final_score}/10.")
            print("   Treat `max_iterations` as a cost cap, not a quality guarantee.")

    print("\n" + "=" * 60)
    print("📋 FINAL OUTPUT:")
    print(current_draft)
    return current_draft, final_score, termination_reason


# --------------------------------------------------------------
print("✅ critique_loop() ready")

#### Run the Loop

In [ ]:
final_output, final_score, termination = await critique_loop(
    topic="Explain what recursion is in programming",
    quality_threshold=8,
    max_iterations=3
)

print(f"\nTermination: {termination} | Final judge score: {final_score}/10")

---

## 🥊 Part 4: True Debate, Two Opposing Positions

Use the critique loop to refine a single output toward a quality bar.

Use a true debate when the question has legitimately opposing positions.

Surface both sides before deciding.

1. Advocate → argues for

2. Skeptic → argues against (independent, same prompt)

3. Advocate → rebuts the skeptic

4. Synthesizer → balanced conclusion

In [ ]:
advocate_agent = Agent(
    name="Advocate",
    instructions="You argue in strong favor of the given position. Make the best case possible. Be specific.",
    model=MODEL
)

skeptic_agent = Agent(
    name="Skeptic",
    instructions="You argue strongly against the given position. Make the strongest opposing case possible. Be specific.",
    model=MODEL
)

synthesis_agent = Agent(
    name="Synthesizer",
    instructions="You synthesize a debate into a balanced conclusion. Extract the strongest points from both sides and produce a nuanced final position.",
    model=REASONING_MODEL
)

# --------------------------------------------------------------
print("✅ Debate agents ready")

#### Run the Debate

In [ ]:
debate_topic = "Microservices are better than monoliths for most software teams"

print("🥊 DEBATE")
print("=" * 60)
print(f"Topic: {debate_topic}\n")

# Round 1: Independent positions, run in parallel since they don't depend on each other
print("Round 1: Independent positions\n")

advocate_result, skeptic_result = await asyncio.gather(
    Runner.run(advocate_agent, input=f"Argue in favor of: {debate_topic}"),
    Runner.run(skeptic_agent, input=f"Argue against: {debate_topic}")
)

advocate_position = advocate_result.final_output
print(f"📢 Advocate:\n{advocate_position}\n")

skeptic_position = skeptic_result.final_output
print(f"🔍 Skeptic:\n{skeptic_position}\n")

# Round 2: Rebuttal
print("Round 2: Rebuttal\n")

rebuttal_input = (
    f"You previously argued: {advocate_position}\n\n"
    f"The skeptic responded: {skeptic_position}\n\n"
    f"Rebut the skeptic's strongest points."
)

rebuttal_result = await Runner.run(advocate_agent, input=rebuttal_input)

rebuttal = rebuttal_result.final_output
print(f"📢 Advocate rebuttal:\n{rebuttal}\n")

# Round 3: Synthesis
print("Round 3: Synthesis\n")

synthesis_input = (
    f"Synthesize this debate into a balanced conclusion.\n\n"
    f"Topic: {debate_topic}\n\n"
    f"Advocate's position:\n{advocate_position}\n\n"
    f"Skeptic's position:\n{skeptic_position}\n\n"
    f"Advocate's rebuttal:\n{rebuttal}"
)

synthesis_result = await Runner.run(synthesis_agent, input=synthesis_input)

print("⚖️ SYNTHESIS")
print("=" * 60)
print(synthesis_result.final_output)
print("=" * 60)


### 💡 Key Takeaway

Both agents receive the same topic and argue independently.

The advocate finds the strongest case for, the skeptic the strongest case against.

The rebuttal prompt asks the advocate to engage with the skeptic's strongest points.

---

## 💪 Practice Exercises

### Exercise 1: Code Review Pipeline

*Covers: proposer/critic pipeline, output chaining*

Build a proposer/critic pipeline where one agent writes code and another reviews it for bugs and improvements.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 1: Code Review Pipeline
# --------------------------------------------------------------
# Objective: Writer writes code, reviewer critiques, writer revises.

code_task = "Write a Python function that finds duplicate values in a list"

# TODO 1: Create a CodeWriter agent with instructions to write clean,
#           well-commented Python functions

# TODO 2: Create a CodeReviewer agent with instructions to check for:
#           - edge cases not handled
#           - performance issues
#           - missing docstring or comments
#           Critic only, do not rewrite

# TODO 3: Run the pipeline:
#           a) CodeWriter writes the function
#           b) CodeReviewer critiques it
#           c) CodeWriter revises based on feedback
#           Print each step with a clear header

# Optional stretch: add a judge-score threshold and max_iterations=2.
#           Return the final score and termination reason.

# --- Write your code below this line ---

### Exercise 2: Business Plan Debate

*Covers: debate pattern, multi-angle critique*

Use the debate pattern to evaluate a business idea from multiple angles.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 2: Business Plan Debate
# --------------------------------------------------------------
# Objective: Apply the full debate pattern to a business idea.

business_idea = "A subscription service that delivers curated book summaries weekly"

# TODO 1: Create a BusinessAdvocate agent, argues strongly in favor of the idea

# TODO 2: Create a BusinessSkeptic agent, argues strongly against the idea

# TODO 3: Create a BusinessAnalyst that produces a viability summary:
#           - strongest evidence for the idea
#           - major risks and assumptions
#           - unanswered questions for a human reviewer

# TODO 4: Run advocate and skeptic in parallel on the same business_idea
#           prompt using asyncio.gather()

# TODO 5: Give the advocate its original position and the skeptic's argument
#           when requesting the rebuttal

# TODO 6: Give the analyst all three outputs (advocate position, skeptic
#           position, and rebuttal). Produce the viability summary for a
#           human reviewer. Print each stage clearly

# --- Write your code below this line ---

---

## 🎯 Key Takeaways

**Separation of roles can improve quality:**

- The writer drafts, the critic identifies weaknesses, the writer revises

- Each agent gets one explicit role rather than many at once
<br>
<br>

**Loops need termination rules:**

- Always set `max_iterations`: a loop without a limit can keep making calls and accumulating cost

- A judge-score threshold can stop the loop early

- Judge scores are signals, not ground truth
<br>
<br>

**Output chaining is the mechanism:**

- `result.final_output` from one agent becomes part of the next agent's input

- No special SDK features needed: just sequential `Runner.run()` calls
<br>
<br>

**Use debate for genuine tradeoffs:**

- Advocate and skeptic argue independently from the same topic

- The rebuttal prompt asks the advocate to answer the strongest opposing points

- The synthesizer receives both positions and produces one conclusion

---

## 📍 Next Step

**Notebook 16: Capstone #2, Multi-Agent Research Team**  

Combine orchestration, parallel execution, built-in tools, and critique into a full multi-agent research pipeline.

---

##### 🔧 [Troubleshooting Guide](https://github.com/barrettscott/openai-agents/blob/main/TROUBLESHOOTING.md#lesson-15-debate-and-critique)

---